# Data Preprocessing Lab: Getting Data ML-Ready

## Student Practice Notebook

**Name:** Aditya Kumar  
**Register Number:** AP24110010333 
**Date:** 2026-08-27  

## Learning Objectives

After completing this lab, you will be able to:

- Handle missing data with justified strategies (not just delete-everything)
- Encode categorical variables into numeric form (label encoding, one-hot encoding)
- Scale/normalize numeric features (min-max scaling, standardization)
- Detect and handle outliers
- Split data into train/test sets correctly
- Apply preprocessing to image data (pixel normalization) and text data (numeric encoding)

### Why this week matters
Every ML model needs **numbers, on a similar scale, with no gaps**. Raw pandas data (mixed types, missing values, unscaled numbers) can't go directly into a model. This lab is the bridge between "clean data" (last week) and "model-ready data" (next steps in your course).

### How to use this notebook
Same as before: **demo → practice → predict-then-run → reflect.**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.precision', 3)

url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
tips = pd.read_csv(url)
tips['tip_pct'] = tips['tip'] / tips['total_bill']
tips.head()


## Part 1: Missing Data — Choosing a Strategy

Last week you dropped or filled missing values without much thought about *which* to choose. This week: which strategy is right depends on the situation.

### Demonstration


In [ ]:
tips_missing = tips.copy()
np.random.seed(42)
missing_idx = np.random.choice(tips_missing.index, size=15, replace=False)
tips_missing.loc[missing_idx, 'total_bill'] = np.nan

print("Missing values:", tips_missing['total_bill'].isna().sum())
print("Percentage missing: {:.1f}%".format(100 * tips_missing['total_bill'].isna().sum() / len(tips_missing)))


**Common strategies:**
- **Drop rows** — safe when missing % is small AND rows are random (not systematically missing)
- **Fill with mean/median** — good for numeric columns; median is safer when outliers exist
- **Fill with mode** — for categorical columns
- **Fill with group-wise mean** — smarter: fill using the mean *within a relevant group* instead of the global mean

### Demonstration: group-wise fill (smarter than global mean)


In [ ]:
# Instead of filling with the overall mean, fill using each day's own mean
# This preserves day-to-day differences instead of flattening everything to one number

day_means = tips_missing.groupby('day')['total_bill'].transform('mean')
tips_filled = tips_missing.copy()
tips_missing_filled_bill = tips_missing['total_bill'].fillna(day_means)
tips_filled['total_bill'] = tips_missing_filled_bill

print("Remaining missing:", tips_filled['total_bill'].isna().sum())


### Student Practice

1. Create missing values in the `tip` column (5 random rows) the same way as the demo.
2. Fill them using the **median** `tip` grouped by `time` (Lunch/Dinner) instead of by day.
3. In 1-2 sentences, justify why you chose group-by-`time` over a global fill for this column.


In [ ]:
# Create 5 random missing values in the tip column
tips_tip_missing = tips.copy()
np.random.seed(42)

missing_tip_idx = np.random.choice(tips_tip_missing.index, size=5, replace=False)
tips_tip_missing.loc[missing_tip_idx, 'tip'] = np.nan

print("Missing tip values:", tips_tip_missing['tip'].isna().sum())

# Fill missing tip values using the median tip within each time group
time_medians = tips_tip_missing.groupby('time')['tip'].transform('median')
tips_tip_filled = tips_tip_missing.copy()
tips_tip_filled['tip'] = tips_tip_missing['tip'].fillna(time_medians)

print("Remaining missing tip values:", tips_tip_filled['tip'].isna().sum())
tips_tip_filled.loc[missing_tip_idx, ['time', 'tip']]


**Your answer:** Group-wise filling can preserve differences between groups that a global mean would hide. For example, lunch and dinner may have different typical tip amounts, so filling a missing tip with the median of its `time` group can be more representative.

## Part 2: Encoding Categorical Variables

ML models need numbers — they can't use `"Male"`, `"Sun"`, `"Yes"` directly. We need to **encode** categories into numeric form.

### 2a. Label Encoding (for ordinal/binary categories)

### Demonstration


In [ ]:
tips_enc = tips.copy()

# Binary categories: map directly to 0/1
tips_enc['sex_encoded'] = tips_enc['sex'].map({'Male': 0, 'Female': 1})
tips_enc['smoker_encoded'] = tips_enc['smoker'].map({'No': 0, 'Yes': 1})

tips_enc[['sex','sex_encoded','smoker','smoker_encoded']].head()


### 2b. One-Hot Encoding (for categories with no natural order)

`day` has 4 categories with no inherent ranking (Thur isn't "less than" Fri). Label encoding (0,1,2,3) would wrongly imply an order. **One-hot encoding** creates a separate 0/1 column per category instead.

### Demonstration


In [ ]:
day_dummies = pd.get_dummies(tips_enc['day'], prefix='day')
tips_enc = pd.concat([tips_enc, day_dummies], axis=1)
tips_enc[['day','day_Thur','day_Fri','day_Sat','day_Sun']].head()


*Your prediction:* Label encoding is appropriate because `time` has only two categories. Mapping Lunch=0 and Dinner=1 gives a simple binary feature without creating unnecessary columns.

In [ ]:
# Prediction: Since time has only two categories, binary label encoding is appropriate.
# Lunch = 0, Dinner = 1
tips_enc['time_encoded'] = tips_enc['time'].map({'Lunch': 0, 'Dinner': 1})

print("Prediction: label encoding is appropriate for a binary category.")
tips_enc[['time', 'time_encoded']].head()


*Your answer:* Label-encoding `day` as 0, 1, 2, and 3 creates an artificial numerical order. A model could incorrectly interpret Sunday as being greater than Saturday or Saturday as being twice the value of another day, even though the days are just categories with no natural ranking.

## Part 3: Feature Scaling

`total_bill` ranges roughly 3-50, while `tip_pct` ranges roughly 0-1. Many ML algorithms (e.g. distance-based ones like KNN, or gradient-based ones) perform poorly when features are on very different scales — a large-range feature can dominate just because of its scale, not because it's more important.

### 3a. Min-Max Scaling (rescales to a fixed 0-1 range)

### Demonstration


In [ ]:
def min_max_scale(series):
    return (series - series.min()) / (series.max() - series.min())

tips_enc['total_bill_scaled'] = min_max_scale(tips_enc['total_bill'])
tips_enc[['total_bill','total_bill_scaled']].describe()


### 3b. Standardization (rescales to mean=0, std=1)

### Demonstration


In [ ]:
def standardize(series):
    return (series - series.mean()) / series.std()

tips_enc['total_bill_standardized'] = standardize(tips_enc['total_bill'])
tips_enc[['total_bill','total_bill_standardized']].describe()


### Student Practice — predict then check

Predict: after standardizing, what will the **mean** of `total_bill_standardized` be (approximately)? What about the standard deviation? Write your prediction, then verify with `.mean()` and `.std()`.

*Your prediction:*

Now apply **min-max scaling** to the `size` column and store it as `size_scaled`.


In [ ]:
# Prediction: mean ≈ 0 and standard deviation ≈ 1
print("Predicted mean: approximately 0")
print("Predicted standard deviation: approximately 1")
print("Actual mean:", tips_enc['total_bill_standardized'].mean())
print("Actual standard deviation:", tips_enc['total_bill_standardized'].std())

# Apply min-max scaling to size
tips_enc['size_scaled'] = min_max_scale(tips_enc['size'])

tips_enc[['size', 'size_scaled']].head()


*Your answer:* Min-max scaling maps values to a fixed range, usually 0 to 1. Standardization centers values around a mean of 0 and scales them by their standard deviation. Min-max scaling is useful when a fixed range is desired, while standardization is often useful for models that work better with centered features. Both can be affected by extreme outliers, but min-max scaling can be especially compressed by them.

## Part 4: Outlier Detection and Handling

Outliers are unusually extreme values that can distort model training. A common rule: values beyond **1.5 × IQR** (interquartile range) from Q1/Q3 are considered outliers.

### Demonstration


In [ ]:
Q1 = tips['total_bill'].quantile(0.25)
Q3 = tips['total_bill'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = tips[(tips['total_bill'] < lower_bound) | (tips['total_bill'] > upper_bound)]
print(f"Bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")
print(f"Number of outliers: {len(outliers)}")
outliers[['total_bill','day','size']]


In [ ]:
# Visualize outliers with a boxplot
plt.figure()
plt.boxplot(tips['total_bill'])
plt.title("Total Bill — Outlier Check")
plt.ylabel("Total Bill")
plt.show()


### Student Practice

1. Run the same IQR outlier check on the `tip` column.
2. Instead of dropping outliers, **cap** them: replace any value above `upper_bound` with `upper_bound` itself, and any value below `lower_bound` with `lower_bound` (this is called "capping" or "winsorizing" — it keeps the row but limits the extreme value).


In [ ]:
# IQR outlier check for the tip column
Q1_tip = tips['tip'].quantile(0.25)
Q3_tip = tips['tip'].quantile(0.75)
IQR_tip = Q3_tip - Q1_tip

lower_bound_tip = Q1_tip - 1.5 * IQR_tip
upper_bound_tip = Q3_tip + 1.5 * IQR_tip

tip_outliers = tips[
    (tips['tip'] < lower_bound_tip) |
    (tips['tip'] > upper_bound_tip)
]

print(f"Tip bounds: [{lower_bound_tip:.2f}, {upper_bound_tip:.2f}]")
print(f"Number of tip outliers: {len(tip_outliers)}")

# Cap the outliers instead of dropping the rows
tips_capped = tips.copy()
tips_capped['tip_capped'] = tips_capped['tip'].clip(
    lower=lower_bound_tip,
    upper=upper_bound_tip
)

print("Original tip range:", tips['tip'].min(), "-", tips['tip'].max())
print("Capped tip range:", tips_capped['tip_capped'].min(), "-", tips_capped['tip_capped'].max())

tips_capped[['tip', 'tip_capped']].head()


*Your answer:* Capping keeps the original row instead of throwing away potentially useful information, while limiting the influence of extreme values. Dropping is better when an outlier is clearly caused by an error or when the extreme observation is not representative of the population and would harm the analysis.

## Part 5: Train/Test Split

Before training any model, data must be split so we can test performance on data the model has never seen. Splitting **after** all the preprocessing above ensures both sets are equally clean and encoded.

### Demonstration


In [ ]:
from sklearn.model_selection import train_test_split

X = tips_enc[['total_bill_scaled','size','sex_encoded','smoker_encoded']]
y = tips_enc['tip_pct']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


### Student Practice

1. Redo the split with `test_size=0.3` instead of 0.2. How many rows end up in the test set now?
2. Why do we set `random_state=42` (or any fixed number)? What would change if you removed it and ran the split twice?


In [ ]:
# Redo the train/test split using 30% of the data for testing
X_train_30, X_test_30, y_train_30, y_test_30 = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("Train shape:", X_train_30.shape)
print("Test shape:", X_test_30.shape)
print("Number of test rows:", len(X_test_30))

print(
    "random_state=42 makes the split reproducible. "
    "Without it, repeated runs can produce different train/test rows."
)


*Your answer:* The test set should represent unseen data. If the model is trained using the test set, its performance estimate can become overly optimistic because the model has already had access to those examples. Splitting first helps measure how well the model generalizes to new data.

# Mini Project A: Image Preprocessing Pipeline (Image Track)

Raw image pixel values (0-255) are rarely fed directly into a model — they're almost always normalized first. You'll now apply real preprocessing to the images from your Week 2 mini-project.

## Step 1: Normalize pixel values

### Demonstration


In [ ]:
# Upload your image files when running this notebook in Google Colab.
# The cells below will preprocess every uploaded image.
from google.colab import files
uploaded = files.upload()


In [ ]:
from PIL import Image
import numpy as np

filenames = list(uploaded.keys())
sample_name = filenames[0]
img_array = np.array(Image.open(sample_name))

print("Original pixel range:", img_array.min(), "-", img_array.max())
print("Original dtype:", img_array.dtype)

# Normalize to 0-1 range (standard preprocessing step before feeding into a CNN)
img_normalized = img_array / 255.0

print("Normalized pixel range:", img_normalized.min(), "-", img_normalized.max())
print("Normalized dtype:", img_normalized.dtype)


### Student Practice

1. Normalize **all** your uploaded images (loop over `filenames`) and store each normalized array in a list.
2. Build a small pandas DataFrame with columns `filename`, `min_pixel_before`, `max_pixel_before`, `min_pixel_after`, `max_pixel_after` to confirm normalization worked correctly across all images.


In [ ]:
# Normalize every uploaded image and store the arrays in a list
normalized_images = []
image_stats = []

for filename in filenames:
    image_array = np.array(Image.open(filename))
    normalized_array = image_array / 255.0
    normalized_images.append(normalized_array)

    image_stats.append({
        'filename': filename,
        'min_pixel_before': image_array.min(),
        'max_pixel_before': image_array.max(),
        'min_pixel_after': normalized_array.min(),
        'max_pixel_after': normalized_array.max()
    })

normalization_df = pd.DataFrame(image_stats)
normalization_df


## Step 2: Resize images to a consistent shape

Models need every input image to be the **same size**. Your uploaded images likely have different dimensions. Resizing standardizes this.

### Demonstration


In [ ]:
target_size = (128, 128)

img_pil = Image.open(sample_name)
img_resized = img_pil.resize(target_size)
img_resized_array = np.array(img_resized)

print("Original shape:", img_array.shape)
print("Resized shape:", img_resized_array.shape)


### Student Practice

Resize all your uploaded images to `(128, 128)`, normalize each to 0-1, and confirm every resulting array has the exact same shape using a pandas DataFrame (columns: `filename`, `shape`).


In [ ]:
# Resize all uploaded images to 128x128, then normalize to 0-1
target_size = (128, 128)

resized_normalized_images = []
resize_stats = []

for filename in filenames:
    image = Image.open(filename)
    resized_image = image.resize(target_size)
    resized_array = np.array(resized_image)
    normalized_array = resized_array / 255.0

    resized_normalized_images.append(normalized_array)

    resize_stats.append({
        'filename': filename,
        'shape': normalized_array.shape
    })

resize_df = pd.DataFrame(resize_stats)
resize_df


*Your answer:* A model expects inputs with a consistent number of dimensions and pixels. If images have different shapes, they cannot be stacked into one regular input tensor, and a model with a fixed input shape cannot process them consistently.

# Mini Project B: Text Preprocessing Pipeline (NLP Track)

Just like images, raw text needs preprocessing before a model can use it: cleaning, and converting words into numeric IDs (since bag-of-words counts alone aren't how most modern NLP pipelines represent text).

## Step 1: Text cleaning

### Demonstration


In [ ]:
raw_text = "NumPy, Pandas, and Scikit-Learn are AMAZING tools!!! I use NumPy every day."

# Cleaning steps: lowercase, remove punctuation, tokenize
import re

cleaned = raw_text.lower()
cleaned = re.sub(r'[^a-z0-9\s]', '', cleaned)  # remove punctuation
tokens = cleaned.split()

print("Before:", raw_text)
print("After:", tokens)


### Student Practice

Write your own messy sentence (include punctuation, capital letters, maybe a number). Clean it using the same pattern and print the resulting tokens.


In [ ]:
# My own messy sentence
my_text = "Python, DATA Science 2026 is AMAZING!!!"

# Clean: lowercase, remove punctuation, tokenize
my_cleaned = my_text.lower()
my_cleaned = re.sub(r'[^a-z0-9\s]', '', my_cleaned)
my_tokens = my_cleaned.split()

print("Before:", my_text)
print("After:", my_tokens)


## Step 2: Encoding words as numeric IDs

Models need numbers, not words. We'll build a **word-to-ID mapping** (a simple form of what's called a "vocabulary index" — the first step toward embeddings).

### Demonstration


In [ ]:
vocabulary = sorted(set(tokens))
word_to_id = {word: idx for idx, word in enumerate(vocabulary)}

print("Word to ID mapping:", word_to_id)

encoded_tokens = [word_to_id[word] for word in tokens]
print("Original tokens:", tokens)
print("Encoded as IDs:  ", encoded_tokens)


### Student Practice

1. Build a `word_to_id` mapping for the tokens from your Step 1 sentence.
2. Encode your tokens into a list of numeric IDs.
3. Build a pandas DataFrame with columns `word` and `id` showing the full mapping, sorted by `id`.


In [ ]:
# Build the vocabulary and word-to-ID mapping
my_vocabulary = sorted(set(my_tokens))
my_word_to_id = {
    word: idx for idx, word in enumerate(my_vocabulary)
}

# Encode the tokens
my_encoded_tokens = [my_word_to_id[word] for word in my_tokens]

# Build a DataFrame sorted by ID
word_id_df = pd.DataFrame(
    list(my_word_to_id.items()),
    columns=['word', 'id']
).sort_values('id').reset_index(drop=True)

print("Word to ID mapping:", my_word_to_id)
print("Original tokens:", my_tokens)
print("Encoded tokens:", my_encoded_tokens)
word_id_df


*Your answer:* Yes, the same misleading-order problem can happen. If words receive IDs such as apple=0, banana=1, and zebra=2, a model might treat those numbers as meaningful magnitudes or distances even though the IDs are only labels. This is why real NLP models commonly use representations such as one-hot vectors or learned embeddings rather than treating word IDs as continuous numeric values.

# Final Self-Check

- [x] I handled missing data using a justified strategy, not just dropped everything blindly
- [x] I correctly chose label encoding vs one-hot encoding based on whether categories have order
- [x] I scaled numeric features and can explain the difference between min-max and standardization
- [x] I detected outliers using IQR and handled them (capped, not just deleted)
- [x] I split data into train/test sets correctly
- [x] I completed Mini Project A (image normalization/resizing) or B (text cleaning/encoding)
- [x] I answered all reflection questions in my own words

## Final Reflection Questions

### 1. Put the preprocessing steps in this lab in the order you'd typically apply them to a raw dataset, and briefly justify the order.

A typical order is: **inspect and clean the data → handle missing values → encode categorical variables → detect and handle outliers → scale numeric features → split into training and test sets before fitting the model**. In a production ML workflow, the split should actually happen before fitting preprocessing statistics such as means, medians, or scaling parameters, so that information from the test set does not leak into the training process.

### 2. Why is preprocessing considered part of "data work," and not something a model does automatically?

Preprocessing converts raw, inconsistent data into a representation that a model can use. Models generally do not automatically know how to handle missing values, categorical strings, different numeric scales, unusual outliers, image dimensions, or raw text in the way the task requires. Good preprocessing also requires decisions based on the meaning and quality of the data.

### 3. Compare preprocessing images (Mini Project A) vs preprocessing text (Mini Project B). What's conceptually similar between normalizing pixels and encoding words as IDs?

Both pipelines convert raw data into a numerical representation suitable for a model. Image preprocessing changes pixel values from a raw 0–255 range into a normalized numerical range and makes image shapes consistent. Text preprocessing cleans the raw characters and converts words into numeric IDs. In both cases, preprocessing creates a consistent representation while preserving useful information for the model.

### 4. What's one preprocessing concept from this lab you still feel unsure about? Be specific.

I am still somewhat unsure about when to choose capping versus dropping outliers. The choice depends on whether an extreme value is a genuine observation or a data-quality error, and on how strongly the model is affected by the extreme values.
